# Calibration And Uncertainty Demo

Fit a synthetic parameter, inspect residuals, propagate uncertainty, and compute local sensitivity using package utilities.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np

from fungal_model.calibration import FittableParameter, fit_least_squares
from fungal_model.core.parameters import Parameter, ParameterSet
from fungal_model.core.units import Q_
from fungal_model.uncertainty import ParameterUncertaintySpec, local_sensitivity, run_monte_carlo, LocalSensitivitySpec

In [ ]:
x = Q_(np.linspace(0.0, 5.0, 11), "second")
observed = {"product": Q_(2.0 * x.magnitude, "mole / liter")}
base = ParameterSet([
    Parameter("slope", "m", 1.0, "mole / liter / second", 0.0, "Synthetic calibration initial value.", "testing", "Notebook demo."),
])
prediction = lambda params: {"product": params.require_quantity("m", "mole / liter / second") * x}
fit = fit_least_squares(
    base_parameters=base,
    fittable_parameters=(FittableParameter(
        symbol="m",
        lower_bound=Parameter("lower slope", "m_lower", 0.0, "mole / liter / second", 0.0, "Synthetic bound.", "testing", "Notebook demo."),
        upper_bound=Parameter("upper slope", "m_upper", 5.0, "mole / liter / second", 0.0, "Synthetic bound.", "testing", "Notebook demo."),
    ),),
    prediction_function=prediction,
    observations=observed,
    calibration_source="Synthetic notebook calibration dataset.",
)
fit.fitted_parameters.get("m").quantity

In [ ]:
mc = run_monte_carlo(
    base_parameters=fit.fitted_parameters,
    uncertainty_specs=(ParameterUncertaintySpec(
        symbol="m",
        distribution="normal",
        standard_deviation=Parameter("slope standard deviation", "m_sd", 0.1, "mole / liter / second", 0.0, "Synthetic uncertainty.", "testing", "Notebook demo."),
        source="Synthetic notebook uncertainty specification.",
    ),),
    prediction_function=prediction,
    n_samples=25,
    random_seed=123,
)
sensitivity = local_sensitivity(
    base_parameters=fit.fitted_parameters,
    sensitivity_specs=(LocalSensitivitySpec(symbol="m"),),
    prediction_function=prediction,
)
(mc.n_successful, sensitivity.ranking[0].symbol)